# Topic: Identity and Access Management - RBAC vs ABAC policy engines, privilege access rules, and PDP/PEP simulators
 
## 1. CORE ACCESS CONTROL MODELS
- **Role-Based Access Control (RBAC)**:
  - Permissions are grouped and associated with specific **Roles** (e.g., `Admin`, `Auditor`, `Guest`).
  - Users are assigned roles, granting them all permissions mapped to those roles.
- **Attribute-Based Access Control (ABAC)**:
  - A more granular model where access permissions are evaluated dynamically at runtime based on **Attributes**:
    1. **Subject Attributes**: User's department, clearance level, IP location.
    2. **Resource Attributes**: Document classification (secret vs public), creation date.
    3. **Environment Attributes**: Current local time, system load.
 
## 2. ARCHITECTURAL PATTERNS: PDP & PEP
- **Policy Decision Point (PDP)**: The engine that evaluates active security policies against request metadata and returns an authorize decision (`Permit` or `Deny`).
- **Policy Enforcement Point (PEP)**: The interface (middleware, API filter) that intercepts access requests, queries the PDP, and enforces the block or release action.
 
---


In [ ]:
class PolicyDecisionPoint:
    """PDP Engine that evaluates dynamic RBAC and ABAC rules."""
    def __init__(self):
        # RBAC Roles definition: role -> set of allowed actions
        self.rbac_roles = {
            "Admin": {"create", "read", "update", "delete"},
            "Editor": {"read", "update"},
            "Guest": {"read"}
        }

    def evaluate_access(self, subject, action, resource, environment):
        """PDP Decision Maker. Returns True (Permit) or False (Deny)."""
        user_role = subject.get("role", "Guest")
        
        # 1. Evaluate RBAC Layer
        allowed_actions = self.rbac_roles.get(user_role, set())
        if action not in allowed_actions:
            return False  # Failed RBAC verification
            
        # 2. Evaluate ABAC Layer (Dynamic Rules)
        # Rule: Only allow updates/deletions if the user is in the "Engineering" department
        if action in ("update", "delete"):
            if subject.get("department") != "Engineering":
                return False  # Failed Department Attribute check
                
        # Rule: Prevent access to classification="Secret" resources unless clearance >= 3
        if resource.get("classification") == "Secret":
            if subject.get("clearance_level", 0) < 3:
                return False  # Failed Clearance Attribute check
                
        # Rule: Prevent modifications outside office hours (09:00 to 17:00)
        if action in ("create", "update", "delete"):
            hour = environment.get("current_hour", 12)
            if hour < 9 or hour > 17:
                return False  # Failed Environment time check
                
        return True  # All checks passed!



In [ ]:
# Simulating the Policy Enforcement Point (PEP)
class PolicyEnforcementPoint:
    def __init__(self, pdp):
        self.pdp = pdp

    def handle_request(self, subject, action, resource, environment):
        """PEP Intercepts request and enforces PDP decision."""
        print(f"\n[PEP REQUEST]: User '{subject['id']}' ({subject['role']}) wants to '{action}' resource '{resource['id']}'")
        print(f"  Metadata -> Dept: {subject.get('department')} | Clearance: {subject.get('clearance_level')} | Hour: {environment.get('current_hour')}")
        
        is_permitted = self.pdp.evaluate_access(subject, action, resource, environment)
        if is_permitted:
            print("  [DECISION]: ACCESS PERMITTED.")
            return True
        else:
            print("  [DECISION]: ACCESS DENIED! Policy block active.")
            return False



In [ ]:
# Testing the IAM Engine
pdp_engine = PolicyDecisionPoint()
pep = PolicyEnforcementPoint(pdp_engine)

# Metadata Definitions
admin_user = {"id": "bob_admin", "role": "Admin", "department": "Engineering", "clearance_level": 5}
auditor_user = {"id": "alice_audit", "role": "Editor", "department": "Compliance", "clearance_level": 2}
document_secret = {"id": "core_infrastructure_plans", "classification": "Secret"}
document_public = {"id": "news_release", "classification": "Public"}

env_business_hours = {"current_hour": 14}
env_midnight = {"current_hour": 23}

# Run Scenario tests
# 1. Admin accesses secret document in business hours -> Allowed
pep.handle_request(admin_user, "read", document_secret, env_business_hours)

# 2. Editor accesses secret document -> Blocked (clearance too low)
pep.handle_request(auditor_user, "read", document_secret, env_business_hours)

# 3. Editor updates public document -> Blocked (not in Engineering department)
pep.handle_request(auditor_user, "update", document_public, env_business_hours)

# 4. Admin updates public document at midnight -> Blocked (outside office hours)
pep.handle_request(admin_user, "update", document_public, env_midnight)
